# 04 · Multivariate mode and the cold start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USER/chronos2-in-practice/blob/main/notebooks/04_multivariate_coldstart.ipynb)

Companion notebook for *Chronos-2 in Practice (2/5)*.

Both experiments here are the same mechanism — the group ID — pointed at two different problems.

**What you will learn**

- Whether forecasting two related Spanish series jointly beats forecasting each on its own,
  and why the paper leads you to expect little.
- What `cross_learning=True` is worth for a series with three weeks of history, measured
  against the right control rather than against nothing.
- That the control matters: putting series in the same frame does nothing unless they also
  share a group.
- Whether a smaller, more similar group beats a bigger one.

Runtime: ~10 min on a GPU (FAST_MODE: ~2 min).

In [ ]:
# On Google Colab, clone the companion repo and install it (Colab's own PyTorch is kept).
# Locally, `uv sync` already installed everything and this cell does nothing.
import sys

if "google.colab" in sys.modules:
    !git clone --depth 1 https://github.com/YOUR_GITHUB_USER/chronos2-in-practice.git
    %cd chronos2-in-practice
    %pip install -q .
    import importlib

    importlib.invalidate_caches()

In [ ]:
import time

import pandas as pd
from huggingface_hub.utils import logging as hub_logging
from IPython.display import Image, display
from transformers.utils import logging as transformers_logging

from c2p.config import get_device, get_paths, is_fast_mode, set_seed
from c2p.data import (
    spain_energy_frame,
    spain_weather_frame,
    validate_and_prepare,
)
from c2p.eval import MODEL_COL, BacktestConfig, backtest, evaluate, skill_score, skill_table
from c2p.experiments import FREQ, LOAD, PRICE, weekly_sample_config
from c2p.models import Chronos2Forecaster, load_chronos2
from c2p.plotting import plot_metric_vs_param, save_figure, set_style
from c2p.results import load_results, run_metadata, save_results
from c2p.schema import ID_COL, TARGET_NAME_COL, TIMESTAMP_COL

FAST_MODE = is_fast_mode(default=False)
PATHS = get_paths(fast_mode=FAST_MODE)
set_seed()
DEVICE = get_device()
set_style()
hub_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

for key, value in run_metadata(device=DEVICE, fast_mode=FAST_MODE).items():
    print(f"{key:>20}: {value}")

## 1. Multivariate: load and price together

Spanish demand and day-ahead price are driven by the same system, so they are the obvious
candidate for joint forecasting: `target=["load_actual", "price_actual"]` puts both variates in
one group, and Chronos-2 predicts them together.

The paper sets the expectation low. Its multivariate gains are modest, and Chronos-2 *in
univariate mode* already beats Toto-1.0, a natively multivariate model — the large gains in the
paper come from covariates (article 3), not from joint targets.

This is therefore a quick check, on the 52-window weekly sample rather than the full year, and
scored per target so that the two series are not averaged into one unreadable number. (With
several targets, WQL across targets is dominated by whichever has the larger values — here,
load in MW against price in EUR/MWh.)

In [ ]:
energy, energy_report = validate_and_prepare(
    spain_energy_frame()[[ID_COL, TIMESTAMP_COL, LOAD, PRICE]],
    freq=FREQ,
    target=[LOAD, PRICE],
)
print(energy_report)

pipeline = load_chronos2(device=DEVICE)
chronos = Chronos2Forecaster(pipeline)

In [ ]:
def univariate_metrics(target: str) -> pd.DataFrame:
    """Backtest one target on its own, labelled so it can be compared per target."""
    config = weekly_sample_config(FAST_MODE, target=target)
    frame = energy[[ID_COL, TIMESTAMP_COL, target]]
    metrics = evaluate(backtest(chronos, frame, config), frame, config)
    return metrics.assign(**{TARGET_NAME_COL: target, MODEL_COL: "univariate"})


start = time.perf_counter()
joint_config = weekly_sample_config(FAST_MODE, target=(LOAD, PRICE))
joint = evaluate(
    backtest(chronos, energy, joint_config), energy, joint_config, by_target=True
).assign(**{MODEL_COL: "multivariate"})
separate = pd.concat([univariate_metrics(LOAD), univariate_metrics(PRICE)], ignore_index=True)
multivariate = pd.concat([separate, joint], ignore_index=True)
print(f"Three backtests in {(time.perf_counter() - start) / 60:.1f} min")
print("Saved", save_results(multivariate, "04_multivariate_metrics", PATHS).name)

In [ ]:
multivariate = load_results("04_multivariate_metrics", PATHS)
rows = []
for target in (LOAD, PRICE):
    block = multivariate[multivariate[TARGET_NAME_COL] == target]
    table = skill_table(block.drop(columns=TARGET_NAME_COL), baseline="univariate", metric="WQL")
    rows.append(table[table["model"] == "multivariate"].assign(target=target))
multivariate_skill = pd.concat(rows, ignore_index=True)
print("Saved", save_results(multivariate_skill, "04_multivariate_skill", PATHS).name)

for row in load_results("04_multivariate_skill", PATHS).itertuples():
    direction = "better" if row.skill > 0 else "worse"
    print(
        f"{row.target:<13} multivariate is {abs(row.skill):5.1%} {direction} than univariate "
        f"on a typical window, 95% CI [{row.ci_low:+.1%}, {row.ci_high:+.1%}], "
        f"winning {row.win_rate:.0%} of windows"
    )

A confidence interval that spans zero is the expected outcome here, not a failed experiment:
joint targets are not where Chronos-2's in-context learning pays off. The next section is.

## 2. The cold start: how short does history have to get?

A new store, a new sensor, a new product. The paper's suggestion for this case is to put the
new series in a group with related ones, so the model can borrow their shape. The practical
question is not whether that works in principle but *when it starts to matter* — and that needs
two measurements, not one:

1. **The handicap.** How much worse is a series with `h` hours of history than the same series
   with everything? If the answer is "not at all", there is nothing for grouping to recover.
2. **The gain.** At that same `h`, how much does sharing a group with related series help?

The weather file gives five Spanish cities measured the same way, which is the "related series"
setting. One city is made cold and the other four stay complete, and we sweep the cold city's
history from three days to three weeks.

Two details decide whether this measures anything:

- **A cold start has to stay cold.** Truncating once and then rolling origins forward would
  leave the "cold" city with a year of history by the end. Each origin is its own *episode*: the
  series is re-truncated to `h` hours before that origin, so every window is a genuine
  short-history forecast.
- **Only the cold city may be scored.** Arms that put five cities in the frame produce forecasts
  for five cities, and averaging them would mix the cold start with four easy series. Every arm
  is scored on the cold city alone.

One extra arm, `in_frame`, puts all five cities in the call with **separate** groups. It is the
control: series in different groups cannot see each other, so it has to land exactly on the
`alone` arm. If it does not, something other than grouping is moving the numbers.

In [ ]:
weather, weather_report = validate_and_prepare(
    spain_weather_frame(),
    freq=FREQ,
    target="temp_c",
    duplicates="mean",
    categorical=["weather_main"],
)
weather = weather[[ID_COL, TIMESTAMP_COL, "temp_c"]]
CITIES = sorted(weather[ID_COL].unique())
COLD_CITY = CITIES[0]
HORIZON = 24
HISTORIES = (72, 504) if FAST_MODE else (72, 168, 336, 504)  # hours of history to sweep
N_EPISODES = 6 if FAST_MODE else 52
print(f"Cities: {CITIES}")
print(f"Cold-start city: {COLD_CITY}")
print(f"Histories swept: {HISTORIES} hours ({[h // 24 for h in HISTORIES]} days)")

wide = weather.pivot_table(index=TIMESTAMP_COL, columns=ID_COL, values="temp_c")
correlation = wide.corr()[COLD_CITY].drop(COLD_CITY).sort_values(ascending=False)
print(
    f"Correlation with {COLD_CITY}: "
    + ", ".join(f"{city} {correlation[city]:.2f}" for city in correlation.index)
)

In [ ]:
EPISODE_CUTOFFS = pd.date_range(
    end=weather[TIMESTAMP_COL].max() - pd.Timedelta(hours=HORIZON),
    periods=N_EPISODES,
    freq=pd.Timedelta(weeks=1),
)
EPISODE_CONFIG = BacktestConfig(
    prediction_length=HORIZON, n_windows=1, freq=FREQ, season_length=24, target="temp_c"
)
print(f"{N_EPISODES} episodes, {EPISODE_CUTOFFS[0]} to {EPISODE_CUTOFFS[-1]} UTC")


def episode_frame(cutoff: pd.Timestamp, cities: list[str], history: int | None) -> pd.DataFrame:
    """The frame one episode sees; the cold city keeps only ``history`` hours (None = all)."""
    frame = weather[
        weather[ID_COL].isin(cities)
        & (weather[TIMESTAMP_COL] <= cutoff + pd.Timedelta(hours=HORIZON))
    ]
    if history is not None:
        too_old = (frame[ID_COL] == COLD_CITY) & (
            frame[TIMESTAMP_COL] <= cutoff - pd.Timedelta(hours=history)
        )
        frame = frame[~too_old]
    return frame.reset_index(drop=True)


def run_arm(
    name: str, cities: list[str], cross_learning: bool, history: int | None
) -> pd.DataFrame:
    """Metrics of one arm over every episode, scored on the cold city only."""
    model = Chronos2Forecaster(pipeline, name=name, cross_learning=cross_learning)
    rows = []
    for episode, cutoff in enumerate(EPISODE_CUTOFFS):
        frame = episode_frame(cutoff, cities, history)
        forecasts = backtest(model, frame, EPISODE_CONFIG)
        city_frame = frame[frame[ID_COL] == COLD_CITY].reset_index(drop=True)
        city_forecasts = forecasts[forecasts[ID_COL] == COLD_CITY].reset_index(drop=True)
        rows.append(
            evaluate(city_forecasts, city_frame, EPISODE_CONFIG).assign(
                window=episode, cutoff=cutoff
            )
        )
    return pd.concat(rows, ignore_index=True)

In [ ]:
start = time.perf_counter()
runs = {"full_alone": run_arm("full_alone", [COLD_CITY], False, None)}
for history in HISTORIES:
    runs[f"alone_{history}"] = run_arm(f"alone_{history}", [COLD_CITY], False, history)
    runs[f"cross_{history}"] = run_arm(f"cross_{history}", CITIES, True, history)
shortest = min(HISTORIES)
runs[f"in_frame_{shortest}"] = run_arm(f"in_frame_{shortest}", CITIES, False, shortest)

coldstart = pd.concat(runs.values(), ignore_index=True)
print(f"{len(runs)} arms x {N_EPISODES} episodes in {(time.perf_counter() - start) / 60:.1f} min")
print("Saved", save_results(coldstart, "04_coldstart_metrics", PATHS).name)
for name, group in coldstart.groupby(MODEL_COL, sort=False):
    print(
        f"  {name:<16} WQL {group['WQL'].mean():.4f}  "
        f"(series scored per window: {group['n_series'].iloc[0]:.0f})"
    )

Both measurements come from the same paired windows. The *handicap* compares a short-history
arm with the full-history one; the *gain* compares cross-learning with going alone at the same
history length. Negative handicap means the short series is worse, positive gain means grouping
helps.

In [ ]:
coldstart = load_results("04_coldstart_metrics", PATHS)
rows = []
for history in HISTORIES:
    alone, cross = f"alone_{history}", f"cross_{history}"
    handicap = skill_score(coldstart, model=alone, baseline="full_alone", metric="WQL")
    gain = skill_score(coldstart, model=cross, baseline=alone, metric="WQL")
    for measure, score in (("Cost of a short history", handicap), ("Cross-learning gain", gain)):
        rows.append(
            {
                "history_hours": history,
                "measure": measure,
                "skill": score.skill,
                "ci_low": score.ci_low,
                "ci_high": score.ci_high,
                "win_rate": score.win_rate,
            }
        )
coldstart_sweep = pd.DataFrame(rows)
print("Saved", save_results(coldstart_sweep, "04_coldstart_sweep", PATHS).name)

control = skill_score(
    coldstart, model=f"in_frame_{shortest}", baseline=f"alone_{shortest}", metric="WQL"
)
print(
    f"Control at {shortest} h -- five series in separate groups vs the series alone: "
    f"{control.skill:+.2%} (must be ~0: separate groups cannot see each other)"
)

In [ ]:
coldstart_sweep = load_results("04_coldstart_sweep", PATHS)
gain = coldstart_sweep[coldstart_sweep["measure"] == "Cross-learning gain"].set_index(
    "history_hours"
)
cost = coldstart_sweep[coldstart_sweep["measure"] == "Cost of a short history"].set_index(
    "history_hours"
)
shortest_h, longest_h = min(HISTORIES), max(HISTORIES)
title = (
    f"At {shortest_h} h of history the short series costs {-cost.loc[shortest_h, 'skill']:.0%} "
    f"and a shared group returns {gain.loc[shortest_h, 'skill']:+.0%}"
)
ax = plot_metric_vs_param(
    coldstart_sweep,
    x="history_hours",
    y="skill",
    hue="measure",
    band=("ci_low", "ci_high"),
    colors={},
    logx=True,
    title=title,
    xlabel="Hours of history the cold series has",
    ylabel="WQL skill score",
)
ax.axhline(0, color="#c3c2b7", linewidth=1.0, zorder=1)
ax.yaxis.set_major_formatter(lambda value, _: f"{value:+.0%}")
(figure_path,) = save_figure(ax.figure, "04_coldstart_sweep", PATHS.figures)
display(Image(filename=figure_path, width=800))
print(coldstart_sweep.round(4).to_string(index=False))

## 3. Summary

- Joint targets (`target=[...]`) are the weakest of the three group modes on this data, which
  matches the paper: the model's advantage lives in covariates, not in multivariate targets.
- The cold start is where the group ID earns its place, and the `cold_in_frame` control is what
  makes that claim defensible: series have to *share a group*, not merely share a call.
- Group composition is itself a knob. Whether the tighter group of the two most correlated
  cities beats all four is printed above — the paper leaves retrieval-based grouping as future
  work, and this is a two-line version of it.

**Next:** article 3 and [`05_covariates`](05_covariates.ipynb) — the mode where the paper
reports its largest gains.